# mine-tuning-model

## Install dependencies

Install fine-tuning packages into the current notebook kernel environment.

After installation, restart the kernel and continue from the CUDA check cell.

In [58]:
import sys
print(sys.executable)

from trl import SFTTrainer, SFTConfig
print("trl ok")


c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\python.exe
trl ok


In [59]:
%pip install --upgrade pip
%pip install -r requirements-ft.txt

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.


In [60]:
from pathlib import Path
import os
import torch

# 로컬 실행 기준 프로젝트 폴더 설정
# - 노트북을 실행하는 현재 작업 디렉터리를 기준으로 outputs 폴더를 생성합니다.
PROJECT_DIR = Path.cwd()
RUN_NAME = "minecraft-qwen7b-4bit-lora-full-r8-b4"
OUTPUT_DIR = PROJECT_DIR / "outputs" / RUN_NAME
ADAPTER_SAVE_PATH = PROJECT_DIR / "outputs" / f"{RUN_NAME}-adapter"
REQUIREMENTS_PATH = PROJECT_DIR / "requirements-ft.txt"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

# 로컬 CUDA/GPU 확인
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU가 감지되지 않았습니다. Qwen2.5 LoRA 학습은 NVIDIA GPU 환경을 권장합니다.")

# GPU에 따라 bf16/fp16 자동 선택
USE_BF16 = torch.cuda.is_bf16_supported()
USE_FP16 = not USE_BF16
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print(f"프로젝트 폴더: {PROJECT_DIR}")
print(f"학습 결과 저장 폴더: {OUTPUT_DIR}")
print(f"LoRA 어댑터 저장 폴더: {ADAPTER_SAVE_PATH}")
print("양자화 모드: 4bit")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"bf16 사용: {USE_BF16}, fp16 사용: {USE_FP16}")

프로젝트 폴더: c:\Users\SSAFY\Desktop\rladbcks\mine-tuning-model
학습 결과 저장 폴더: c:\Users\SSAFY\Desktop\rladbcks\mine-tuning-model\outputs\minecraft-qwen7b-4bit-lora-full-r8-b4
LoRA 어댑터 저장 폴더: c:\Users\SSAFY\Desktop\rladbcks\mine-tuning-model\outputs\minecraft-qwen7b-4bit-lora-full-r8-b4-adapter
양자화 모드: 4bit
GPU: NVIDIA GeForce RTX 5060 Ti
bf16 사용: True, fp16 사용: False


In [61]:
# Optional: uncomment only if you want to snapshot the current local environment.
# Do not overwrite requirements-ft.txt; keep it as the GitHub install file.
#
# import subprocess
# import sys
# LOCAL_REQUIREMENTS_PATH = PROJECT_DIR / "requirements-ft-local.txt"
# with LOCAL_REQUIREMENTS_PATH.open("w", encoding="utf-8") as f:
#     subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=f, check=True)
# print(f"Local requirements saved: {LOCAL_REQUIREMENTS_PATH}")


# 데이터 로드 및 Instruction 형식 변환

In [62]:
from datasets import load_dataset
import os
import random

# Load dataset
ds = load_dataset("ddorin/minecraft-question-answr-260k")
print(ds)

# Number of samples to use for local fine-tuning.
# Default is the full dataset. Override DATA_SIZE for smaller test runs.
DATA_SIZE_ENV = os.getenv("DATA_SIZE", "full").lower()
DATA_SIZE = len(ds["train"]) if DATA_SIZE_ENV in {"full", "all"} else int(DATA_SIZE_ENV)
SAMPLE_SEED = int(os.getenv("SAMPLE_SEED", "42"))
VALIDATION_RATIO = float(os.getenv("VALIDATION_RATIO", "0"))
print(f"sample seed: {SAMPLE_SEED}")
print(f"data size: {min(DATA_SIZE, len(ds['train']))} / {len(ds['train'])}")
print(f"validation ratio: {VALIDATION_RATIO}")

raw_data = (
    ds["train"]
    .shuffle(seed=SAMPLE_SEED)
    .select(range(min(DATA_SIZE, len(ds["train"]))))
)

# DATA_SIZE=full means all selected data is used for training by default.
# Set VALIDATION_RATIO > 0 if you want to hold out validation data.
if VALIDATION_RATIO > 0:
    split_data = raw_data.train_test_split(test_size=VALIDATION_RATIO, seed=SAMPLE_SEED)
    train_raw = split_data["train"]
    eval_raw = split_data["test"]
else:
    train_raw = raw_data
    eval_raw = None

# Convert to instruction format.
def format_instruction(example):
    question = example["question"]
    answer = example["answer"]

    return {
        "text": f"""<|im_start|>system
You are a beginner-friendly Minecraft guide assistant.
Your job is to explain Minecraft to people who have never played Minecraft before.
Use simple words, step-by-step instructions, and mention required items when useful.
Do not overcomplicate the answer.
<|im_end|>
<|im_start|>user
{question}
<|im_end|>
<|im_start|>assistant
{answer}
<|im_end|>"""
    }

train_data = train_raw.map(
    format_instruction,
    remove_columns=[col for col in ["question", "answer", "source"] if col in train_raw.column_names]
)
eval_data = None
if eval_raw is not None:
    eval_data = eval_raw.map(
        format_instruction,
        remove_columns=[col for col in ["question", "answer", "source"] if col in eval_raw.column_names]
    )

print(f"Data formatted: train={len(train_data)}, validation={0 if eval_data is None else len(eval_data)}")
print("\nSample check:")
print(train_data[0]["text"])


DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'source'],
        num_rows: 268239
    })
})
sample seed: 42
data size: 268239 / 268239
validation ratio: 0.0
Data formatted: train=268239, validation=0

Sample check:
<|im_start|>system
You are a beginner-friendly Minecraft guide assistant.
Your job is to explain Minecraft to people who have never played Minecraft before.
Use simple words, step-by-step instructions, and mention required items when useful.
Do not overcomplicate the answer.
<|im_end|>
<|im_start|>user
{question}
<|im_end|>
<|im_start|>assistant
{answer}
<|im_end|>


# 모델, 토크나이저 로드

In [63]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import os
import gc

# Clear leftover GPU memory from previous failed/rerun attempts.
for name in ["trainer", "model"]:
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    try:
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    except torch.cuda.AcceleratorError as e:
        print(f"CUDA cache cleanup skipped: {e}")
        print("Restart the notebook kernel if model loading still fails.")

# 앞의 로컬 환경 설정 셀을 실행하지 않았을 때를 대비한 기본값
COMPUTE_DTYPE = globals().get("COMPUTE_DTYPE", torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)

model_id = os.getenv("MODEL_ID", "Qwen/Qwen2.5-7B-Instruct")

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4bit QLoRA settings. 8bit does not fit on this GPU without slow CPU offload.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    dtype=COMPUTE_DTYPE,
    device_map={"": 0},
    trust_remote_code=True,
)

# Disable cache for training.
model.config.use_cache = False

print(f"✅ 모델 로드 완료!")

if hasattr(model, "hf_device_map"):
    print(f"디바이스 맵: {model.hf_device_map}")
else:
    print("디바이스 맵 정보 없음. 모델은 정상 로드되었습니다.")

Loading weights: 100%|██████████| 339/339 [00:22<00:00, 15.04it/s]


✅ 모델 로드 완료!
디바이스 맵 정보 없음. 모델은 정상 로드되었습니다.


# LoRa 어댑터 설정

In [64]:
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
import gc
import torch

# Prepare the quantized base model for LoRA training.
# If this cell is re-run after LoRA has already been attached, skip wrapping again.
if isinstance(model, PeftModel):
    print("LoRA adapter is already attached. Skipping get_peft_model().")
    model.print_trainable_parameters()
else:
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except torch.cuda.AcceleratorError as e:
            print(f"CUDA cache cleanup skipped: {e}")

    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
    )

    lora_config = LoraConfig(
        r=8,  # Higher rank increases trainable parameters and VRAM use.
        lora_alpha=16,
        target_modules=[
            "q_proj", "k_proj",
            "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
model.config.use_cache = False


trainable params: 20,185,088 || all params: 7,635,801,600 || trainable%: 0.2643


# 학습 실행(SFTTrainer)

In [ ]:
from trl import SFTTrainer, SFTConfig
import gc
import os
import torch

# Defaults for running this cell without re-running the local setup cell.
OUTPUT_DIR = globals().get("OUTPUT_DIR", "./outputs/minecraft-qwen7b-4bit-lora-full-r8-b4")
USE_BF16 = globals().get("USE_BF16", torch.cuda.is_bf16_supported())
USE_FP16 = globals().get("USE_FP16", not USE_BF16)

gc.collect()
if torch.cuda.is_available():
    try:
        torch.cuda.empty_cache()
    except torch.cuda.AcceleratorError as e:
        print(f"CUDA cache cleanup skipped: {e}")
        print("Restart the notebook kernel before creating Trainer.")

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=int(os.getenv("NUM_TRAIN_EPOCHS", "1")),

    # Full-data LoRA settings for a local 16GB VRAM setup.
    # Effective batch size = per_device_train_batch_size * gradient_accumulation_steps.
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=float(os.getenv("LEARNING_RATE", "2e-4")),
    bf16=USE_BF16,
    fp16=USE_FP16,
    gradient_checkpointing=True,

    logging_steps=25,
    eval_strategy="epoch" if eval_data is not None else "no",
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=False,

    warmup_steps=int(os.getenv("WARMUP_STEPS", "10")),
    lr_scheduler_type="cosine",
    report_to="none",

    # More context for Minecraft QA. Lower to 256 if VRAM runs out.
    max_length=int(os.getenv("MAX_SEQ_LENGTH", "384")),
    packing=False,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=training_args,
    processing_class=tokenizer,
)

print("Training started")
trainer.train()


Tokenizing train dataset: 100%|██████████| 268239/268239 [01:41<00:00, 2640.77 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Training started


Step,Training Loss


In [ ]:
from pathlib import Path

# 앞의 로컬 환경 설정 셀을 실행하지 않았을 때를 대비한 기본값
ADAPTER_SAVE_PATH = globals().get("ADAPTER_SAVE_PATH", Path("./outputs/minecraft-qwen7b-4bit-lora-full-r8-b4-adapter"))

# 학습 완료 후 LoRA 어댑터 저장
# - 추론/RAG 노트북에서 이 경로를 불러와 fine-tuned 모델로 사용 가능
adapter_save_path = str(ADAPTER_SAVE_PATH)

trainer.model.save_pretrained(adapter_save_path)
tokenizer.save_pretrained(adapter_save_path)

print(f"✅ LoRA 어댑터 저장 완료: {adapter_save_path}")
